<a href="https://colab.research.google.com/github/doomguy0991/N132/blob/main/CS231N_Lecture2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Table of Contents

- [Section 1: Introduction to Image Classification](#section-1-introduction-to-image-classification)
    - [1.1 The Semantic Gap](#11-the-semantic-gap)
        - [The Tensor Representation](#the-tensor-representation)
    - [1.2 Core Challenges](#12-core-challenges)
    - [1.3 The Data-Driven Approach](#13-the-data-driven-approach)
        - [Pipeline Visualization](#pipeline-visualization)
        - [The API Perspective](#the-api-perspective)
        - [Summary of Section 1](#summary-of-section-1)
- [Section 2: K-Nearest Neighbor (K-NN) Classifier](#section-2-k-nearest-neighbor-k-nn-classifier)
    - [2.1 Concept and Algorithm](#21-concept-and-algorithm)
        - [Computational Complexity](#computational-complexity)
    - [2.2 Distance Metrics](#22-distance-metrics)
        - [L1 (Manhattan) Distance](#l1-manhattan-distance)
        - [L2 (Euclidean) Distance](#l2-euclidean-distance)
    - [2.3 Decision Boundaries and K](#23-decision-boundaries-and-k)
        - [The Problem with 1-Nearest Neighbor ($K=1$)](#the-problem-with-1-nearest-neighbor-k1)
        - [Introducing $K > 1$](#introducing-k--1)
    - [2.4 Why K-NN is Rarely Used for Images](#24-why-k-nn-is-rarely-used-for-images)
        - [Summary of Section 2](#summary-of-section-2)
- [Section 3: Hyperparameter Tuning and Validation](#section-3-hyperparameter-tuning-and-validation)
    - [3.1 Defining Hyperparameters](#31-defining-hyperparameters)
    - [3.2 Data Splits: The Golden Rule](#32-data-splits-the-golden-rule)
        - [The Bad Idea: Evaluate on Training Data](#the-bad-idea-evaluate-on-training-data)
        - [The Slightly Better Idea: Evaluate on Test Data](#the-slightly-better-idea-evaluate-on-test-data)
        - [The Correct Idea: Train, Validation, and Test Splits](#the-correct-idea-train-validation-and-test-splits)
    - [3.3 Cross-Validation](#33-cross-validation)
        - [The K-Fold Algorithm](#the-k-fold-algorithm)
        - [Summary of Section 3](#summary-of-section-3)
- [Section 4: Linear Classifiers](#section-4-linear-classifiers)
    - [4.1 The Parametric Approach Overview](#41-the-parametric-approach-overview)
    - [4.2 The Algebraic Viewpoint](#42-the-algebraic-viewpoint)
    - [4.3 The Visual Viewpoint: Class Templates](#43-the-visual-viewpoint-class-templates)
    - [4.4 The Geometric Viewpoint: Hyperplanes](#44-the-geometric-viewpoint-hyperplanes)
        - [The Bias Trick](#the-bias-trick)
    - [4.5 Hard Cases for Linear Classifiers](#45-hard-cases-for-linear-classifiers)
        - [Summary of Section 4](#summary-of-section-4)
- [Section 5: Loss Functions (Softmax and Cross-Entropy)](#section-5-loss-functions-softmax-and-cross-entropy)
    - [5.1 The Need for Probabilities](#51-the-need-for-probabilities)
    - [5.2 The Softmax Function](#52-the-softmax-function)
    - [5.3 Cross-Entropy Loss](#53-cross-entropy-loss)
    - [5.4 Information Theory Perspective](#54-information-theory-perspective)
    - [5.5 Loss Characteristics and Sanity Checks](#55-loss-characteristics-and-sanity-checks)
        - [Min and Max Values](#min-and-max-values)
        - [The Initialization Sanity Check](#the-initialization-sanity-check)
        - [Summary of Section 5](#summary-of-section-5)


In [ ]:
import os

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Setting up Colab environment...")

    # 1. Clone the repository
    repo_url = "https://github.com/doomguy0991/N132.git"
    # Only clone if the directory doesn't exist yet
    if not os.path.exists("/content/N132"):
        !git clone $repo_url /content/N132

    # 2. Change working directory to the notebook's location so relative paths for libraries work imports
    %cd "/content/N132"

    print("Setup complete. You can now run the rest of the notebook.")
else:
    print("Running locally or out of Colab. No setup needed.")


# Section 1: Introduction to Image Classification

In this section, we define the core task of computer vision: **Image Classification**. We will explore why this seemingly trivial task (for humans) is profoundly difficult for machines, and how the **data-driven approach** provides a scalable framework to solve it.

## 1.1 The Semantic Gap

Given an image and a predefined set of labels (e.g., `{dog, cat, truck, plane}`), the goal of image classification is to assign the correct label to the image. 

While the human cognitive system effortlessly processes the holistic scene, a computer perceives an image purely as a massive grid of numbers. This profound difference in perception is known as the **Semantic Gap**.

### The Tensor Representation
An image is represented as a large 3D tensor of pixel values. For a standard colored image:
- **Resolution**: e.g., $800 \times 600$ pixels (Height $\times$ Width).
- **Channels**: 3 color channels (Red, Green, Blue).
- **Values**: Each pixel value ranges from $0$ to $255$ (8-bit integer).

Thus, the computer perceives this image as a tensor of shape $\mathbf{X} \in \mathbb{R}^{800 \times 600 \times 3}$. The semantic gap refers to the challenge of extracting high-level meaning (e.g., "this is a cat") from these raw, low-level pixel intensities.

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
%matplotlib inline
import numpy as np
import torch
import matplotlib.pyplot as plt

# Let's visualize the "Semantic Gap" by creating a random tensor to represent how a computer sees an image.
np.random.seed(42)
dummy_image_numpy = np.random.randint(0, 256, (10, 10, 3), dtype=np.uint8)

print(f"Numpy Tensor Shape: {dummy_image_numpy.shape}")
print(f"Top-left pixel RGB values: {dummy_image_numpy[0, 0]}")

# Equivalent in PyTorch
dummy_image_torch = torch.randint(0, 256, (3, 10, 10), dtype=torch.uint8) # PyTorch typically uses Channels-First (C, H, W)
print(f"PyTorch Tensor Shape (C, H, W): {dummy_image_torch.shape}")

plt.figure(figsize=(3,3))
plt.imshow(dummy_image_numpy)
plt.title("How the computer sees a 10x10 'image'")
plt.axis('off')
plt.show()

Numpy Tensor Shape: (10, 10, 3)
Top-left pixel RGB values: [102 220 225]
PyTorch Tensor Shape (C, H, W): torch.Size([3, 10, 10])


: 

## 1.2 Core Challenges

If recognizing objects was simply a matter of identifying a specific pattern of numbers, it would be trivial. However, the exact pixel values of an object change drastically under various conditions. A robust image classification model must be invariant to the following challenges:

1.  **Viewpoint Variation:** A single object oriented in different ways will produce completely different pixel matrices. If a camera pans slightly, every single pixel value in the $\mathbf{X}$ tensor changes.
2.  **Illumination Conditions:** Pixel values are a function of both the object's surface material and the light source. A cat in bright sunlight versus a dark room has vastly different numerical representations.
3.  **Deformation:** Many objects (like cats) are not rigid bodies. They can deform and assume highly varied poses.
4.  **Occlusion:** Objects can be partially obscured by other items in the environment. Sometimes only a tail or paw is visible.
5.  **Scale Variation:** Objects can appear arbitrarily large or small depending on their distance from the camera.
6.  **Background Clutter:** Objects may blend into their environment, making them incredibly difficult to distinguish from the background.
7.  **Intra-class Variation:** A "cat" is a broad category. Cats come in different breeds, colors, sizes, and patterns. The algorithm must learn the underlying "essence" of a cat despite these vast physical differences.

> [!NOTE]
> **Key Intuition:** Hardcoding rules (e.g., "find edges, look for pointy ears") fails because these rules are brittle against the variations listed above. We cannot manually account for every possible deformation, lighting condition, and occlusion.

## 1.3 The Data-Driven Approach

Because hardcoding explicit rules for object recognition is unscalable and fragile, we turn to the **Data-Driven Approach** via Machine Learning.

Instead of defining rules for each object, the process is structured as follows:

1.  **Collect a Dataset:** Gather a massive collection of images and manually annotate them with their correct labels.
2.  **Train a Classifier:** Use a machine learning algorithm to learn the mapping from images to labels. This produces a model.
3.  **Evaluate:** Test the classifier on novel, unseen images to assess its generalization capabilities.

### Pipeline Visualization

![Diagram](assets/mermaid_diagram_1.png)

### The API Perspective
We can conceptualize this approach by defining two primary functions:

*   `train(images, labels)` $\rightarrow$ `model`: Takes in training images and outputs a trained model.
*   `predict(model, test_images)` $\rightarrow$ `test_labels`: Takes the model and unseen images, and predicts their labels.

Let's look at how this conceptual API might look in both standard Python/NumPy and PyTorch contexts.

In [ ]:
# --- NUMPY / SCIKIT-LEARN STYLE API STUB ---
class ClassifierNumpy:
    def train(self, images: np.ndarray, labels: np.ndarray):
        """
        Args:
            images: shape (N, H, W, C)
            labels: shape (N,)
        """
        pass # Memorize or learn parameters

    def predict(self, test_images: np.ndarray) -> np.ndarray:
        """
        Args:
            test_images: shape (M, H, W, C)
        Returns:
            predictions: shape (M,)
        """
        return np.zeros(test_images.shape[0])

# --- PYTORCH STYLE API STUB ---
import torch.nn as nn

class ClassifierPyTorch(nn.Module):
    def __init__(self):
        super().__init__()
        # Define learnable layers here
        pass
        
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """
        The forward pass acts similarly to predict(), but usually outputs raw scores (logits).
        Args:
            images: shape (N, C, H, W)
        Returns:
            logits: shape (N, num_classes)
        """
        return torch.zeros((images.shape[0], 10))

---
### Summary of Section 1
*   **Concepts Introduced:** Image Classification task, Semantic Gap, Tensor Representation, Core Challenges (Viewpoint, Illumination, etc.), Data-Driven Approach.
*   **Notation Introduced:** $\mathbf{X}$ (Image Tensor), $N$ (Number of samples), $C$ (Channels), $H$ (Height), $W$ (Width).
*   **Dependencies for Next Section:** Conceptual understanding of train/predict APIs.
*   **Up Next:** We will implement our first data-driven algorithm, the **K-Nearest Neighbor (K-NN) Classifier**, and explore its distance metrics and computational complexity.

# Section 2: K-Nearest Neighbor (K-NN) Classifier

In this section, we build our first concrete implementation of the data-driven approach: the **K-Nearest Neighbor (K-NN)** algorithm. Though rarely used in modern computer vision due to significant drawbacks, K-NN introduces critical machine learning concepts such as **distance metrics**, **hyperparameters**, and **decision boundaries**.

## 2.1 Concept and Algorithm

The K-NN algorithm is beautifully simple. It doesn't attempt to learn a complex mathematical function mapping an image to a label. Instead, it relies on brute-force memorization and comparison.

*   **Train Function:** Simply memorize every single image and its corresponding label in the training dataset.
*   **Predict Function:** Given a new, unseen test image, compare it against *every* training image. Find the top $K$ most similar training images (the "nearest neighbors"), and have them vote on the label. The majority vote becomes the prediction.

### Computational Complexity
Let's analyze the time complexity, assuming we have $N$ training examples.

*   **Training Time:** $O(1)$ — We are simply copying data into memory. No actual computation happens.
*   **Prediction Time:** $O(N)$ — For *each* new test image, we must calculate the distance to all $N$ training images.

> [!WARNING]
> **Major Pitfall:** This complexity is completely backwards for real-world applications! In practice, we want models that are extremely fast at prediction time (e.g., real-time inference on a self-driving car). We don't care if training takes a week in an offline data center, but prediction must be instant. K-NN fails this requirement entirely.

In [ ]:
import numpy as np
import torch

class NearestNeighborNumpy:
    def __init__(self):
        pass
        
    def train(self, X: np.ndarray, y: np.ndarray):
        """
        X is N x D where each row is an example. Y is 1-dimension of size N
        O(1) Training time!
        """
        self.X_train = X
        self.y_train = y
        
    def predict(self, X: np.ndarray, distance_metric='L1'):
        """
        O(N) Prediction time!
        """
        num_test = X.shape[0]
        Ypred = np.zeros(num_test, dtype=self.y_train.dtype)
        
        # In a real implementation, we would vectorize this. 
        # But for intuition, here is the explicit loop:
        for i in range(num_test):
            if distance_metric == 'L1':
                distances = np.sum(np.abs(self.X_train - X[i,:]), axis=1)
            else: # L2
                distances = np.sqrt(np.sum(np.square(self.X_train - X[i,:]), axis=1))
                
            min_index = np.argmin(distances)
            Ypred[i] = self.y_train[min_index]
            
        return Ypred

# PyTorch Equivalent (utilizing GPU for faster distance computation if available)
class NearestNeighborPyTorch:
    def __init__(self):
        self.X_train = None
        self.y_train = None
        
    def train(self, X: torch.Tensor, y: torch.Tensor):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.X_train = X.to(self.device)
        self.y_train = y.to(self.device)
        
    def predict(self, X: torch.Tensor):
        X = X.to(self.device)
        num_test = X.shape[0]
        Ypred = torch.zeros(num_test, dtype=self.y_train.dtype, device=self.device)
        
        # Efficient vectorized L1 distance using PyTorch broadcasting
        for i in range(num_test):
            distances = torch.sum(torch.abs(self.X_train - X[i,:]), dim=1)
            min_index = torch.argmin(distances)
            Ypred[i] = self.y_train[min_index]
            
        return Ypred.cpu()

## 2.2 Distance Metrics

To find the "nearest" neighbors, we need a mathematical function to measure the similarity (or distance) between two images. Because images are tensors, we can compare them pixel-by-pixel.

Let $I_1$ and $I_2$ be two image tensors (flattened into vectors for simplicity).

### L1 (Manhattan) Distance
The L1 distance calculates the absolute difference between individual pixels and sums them up.

$$ d_1(I_1, I_2) = \sum_{p} |I_1^p - I_2^p| $$

**Geometric Interpretation:** Imagine walking on a grid-like street network (like Manhattan). The distance is the sum of the horizontal and vertical steps. All points that have an equal L1 distance from the origin form a diamond/square shape.

### L2 (Euclidean) Distance
The L2 distance calculates the squared difference between pixels, sums them up, and takes the square root.

$$ d_2(I_1, I_2) = \sqrt{\sum_{p} (I_1^p - I_2^p)^2} $$

**Geometric Interpretation:** This is the straight-line distance ("as the crow flies"). All points that have an equal L2 distance from the origin form a perfect circle.

> [!TIP]
> **When to use which?**
> *   **L1 Distance** is highly dependent on the choice of the coordinate system. If you rotate the feature space, the L1 distance between points changes. It is better when your input features have distinct, individual meanings (e.g., height, weight, salary).
> *   **L2 Distance** is rotationally invariant. It doesn't care about the coordinate axes. It is generally preferred when features are arbitrary or interchangeable (like pixels in an image).

## 2.3 Decision Boundaries and K

When we map out the regions in space where the algorithm would predict class A vs class B, we form **Decision Boundaries**.

### The Problem with 1-Nearest Neighbor ($K=1$)
If we strictly take the single nearest neighbor, our decision boundaries will warp sharply around outlying training data points (noise). 

For example, if a single anomalous yellow dot is surrounded entirely by green dots, a $K=1$ classifier will carve out a tiny, jagged yellow island in the middle of a green ocean. This is **overfitting** to noise.

### Introducing $K > 1$
To make the classifier more robust, we increase $K$. We ask the $K$ closest points to vote. 
*   This smooths out the decision boundaries.
*   The isolated noisy yellow dot will be outvoted by its green neighbors.
*   **Drawback:** It creates "white regions" (ties) where no clear majority exists.

![Diagram](assets/mermaid_diagram_2.png)

## 2.4 Why K-NN is Rarely Used for Images

Despite its pedagogical value, K-NN is almost never used for image classification in practice. 

1.  **Terrible Prediction Speed:** As discussed, $O(N)$ inference time is unacceptable.
2.  **Curse of Dimensionality:** In high-dimensional spaces (like a 3072-dimensional image vector), the concept of "distance" becomes unintuitive. The space is vast, and to densely cover it with training examples to make nearest neighbor work requires an exponentially large dataset.
3.  **Pixel Distance is Meaningless:** L1 and L2 distances on raw pixels do not correspond to perceptual similarity.
    *   Shifting an image 1 pixel to the right creates a massive L2 distance, even though the image is perceptually identical to humans.
    *   Changes in background color or slight lighting shifts dominate the pixel-wise distance, ignoring the actual semantic object.

In the next sections, we will introduce methods to automatically find the best values for $K$ and the distance metric (Hyperparameter Tuning), before moving onto more powerful parametric models.

---
### Summary of Section 2
*   **Concepts Introduced:** K-NN Algorithm, L1/L2 Distance Metrics, Decision Boundaries, Overfitting to noise, Curse of Dimensionality.
*   **Equations Derived:** L1 Distance ($d_1(I_1, I_2) = \sum |I_1 - I_2|$), L2 Distance ($d_2(I_1, I_2) = \sqrt{\sum (I_1 - I_2)^2}$).
*   **Dependencies for Next Section:** Understanding that $K$ and the distance metric are choices we must make, which naturally leads to Hyperparameter Tuning.

# Section 3: Hyperparameter Tuning and Validation

In the previous section on the K-Nearest Neighbor algorithm, we identified two critical choices that needed to be made before we could run the classifier: the value of $K$ (number of neighbors) and the choice of the distance metric (L1 vs L2). 

These choices are not learned from the training data via an optimization algorithm; rather, they are set by the researcher ahead of time. These are known as **Hyperparameters**. In this section, we discuss the proper methodology for selecting them.

## 3.1 Defining Hyperparameters

A machine learning algorithm typically has two types of parameters:
1.  **Learnable Parameters:** Weights and biases that the algorithm learns directly from the data during the training process (we will see these in Section 4).
2.  **Hyperparameters:** Settings that control the learning process itself. They must be chosen *before* training begins.

For K-NN, the hyperparameters are:
*   $K$: How many neighbors should vote?
*   **Distance Metric:** L1, L2, or something else (like cosine similarity)?

> [!IMPORTANT]
> Hyperparameters are incredibly problem-dependent and dataset-dependent. There is rarely a single "best" hyperparameter setting that works for all tasks. Therefore, we must *search* for the best settings using our data.

## 3.2 Data Splits: The Golden Rule

How do we figure out which hyperparameters are best? 

### The Bad Idea: Evaluate on Training Data
If we evaluate our hyperparameters on the same data we used to train the model, we will fail catastrophically.
For K-NN, if we evaluate on the training set, $K=1$ will *always* achieve perfect 100% accuracy because the nearest neighbor to any training image is the image itself. However, a $K=1$ model will likely overfit and generalize poorly to unseen data.

### The Slightly Better Idea: Evaluate on Test Data
We could hold out a portion of our data as a "Test Set," train on the rest, and pick the hyperparameters that give the highest accuracy on the Test Set.
**Why is this bad?** It violates the fundamental purpose of the test set. The test set must remain completely unseen until the very end of the project to provide an unbiased estimate of generalization performance. If you use it to tune hyperparameters, you are implicitly "training" on the test set (overfitting to the test set).

### The Correct Idea: Train, Validation, and Test Splits
The golden standard in machine learning is a three-way split:

1.  **Training Set:** Used exclusively to train the algorithm (memorize the data, in K-NN's case).
2.  **Validation Set (or Dev Set):** A fake test set. Used exclusively to evaluate different hyperparameter settings. You train on the Training Set, evaluate on the Validation Set, and pick the hyperparameters that yield the highest validation accuracy.
3.  **Test Set:** Locked away until the very end. Used only once to report the final, unbiased performance of the chosen model.

![Diagram](assets/mermaid_diagram_3.png)

In [ ]:
import numpy as np

# Conceptual representation of a dataset
np.random.seed(42)
num_samples = 1000
X_full = np.random.randn(num_samples, 100) # 1000 samples, 100 features
y_full = np.random.randint(0, 10, num_samples) # 10 classes

# 1. Shuffle the data
indices = np.arange(num_samples)
np.random.shuffle(indices)
X_full = X_full[indices]
y_full = y_full[indices]

# 2. Split the data (e.g., 80% train, 10% val, 10% test)
num_train = 800
num_val = 100
num_test = 100

X_train, y_train = X_full[:num_train], y_full[:num_train]
X_val, y_val = X_full[num_train:num_train+num_val], y_full[num_train:num_train+num_val]
X_test, y_test = X_full[-num_test:], y_full[-num_test:]

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}\n")

# Pseudo-code for hyperparameter tuning loop:
best_k = 1
best_val_acc = 0.0

for k in [1, 3, 5, 7, 10]:
    # 1. Train on X_train (For K-NN, this is just memorizing)
    # 2. Predict on X_val using k
    # 3. Calculate validation accuracy
    val_acc = np.random.rand() # Simulated accuracy for illustration
    print(f"Testing K={k} -> Val Accuracy: {val_acc:.2f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_k = k

print(f"\nSelected best hyperparameter: K={best_k}")
# Finally, evaluate ONLY ONCE on the test set (X_test, y_test) using best_k

## 3.3 Cross-Validation

In scenarios where the dataset is extremely small, setting aside a dedicated validation set might leave you with too little data to properly train the model. Furthermore, evaluating on a tiny validation set might be highly sensitive to the specific random split.

To mitigate this, we use **K-Fold Cross-Validation** (Note: The "K" here refers to folds, unrelated to the "K" in K-NN).

### The K-Fold Algorithm
Instead of a single, static validation set:
1.  Set aside the Test Set as usual.
2.  Divide the remaining Training Data into $F$ equal partitions (folds).
3.  Iterate $F$ times. For each iteration $i$:
    *   Use fold $i$ as the Validation Set.
    *   Use the remaining $F-1$ folds combined as the Training Set.
    *   Train the model and record the validation accuracy.
4.  Average the $F$ validation accuracies to get a single, robust performance estimate for that specific hyperparameter setting.

![Diagram](assets/mermaid_diagram_4.png)

> [!NOTE]
> Cross-validation provides a much more statistically reliable estimate of generalization performance. However, because it requires training the model $F$ times for *every* hyperparameter combination, it is computationally expensive. It is heavily used for small datasets and non-parametric algorithms like K-NN, but is generally avoided in deep learning on massive datasets (like ImageNet) due to the extreme computational cost.

---
### Summary of Section 3
*   **Concepts Introduced:** Hyperparameters vs. Learnable Parameters, Train/Validation/Test Splits, Data Leakage (Overfitting to the test set), K-Fold Cross-Validation.
*   **Notation Introduced:** $F$ (Number of folds in cross-validation).
*   **Dependencies for Next Section:** We have concluded the non-parametric paradigm. We will now shift gears entirely into parametric models where parameters are mathematically optimized rather than explicitly searched.

# Section 4: Linear Classifiers

In the previous sections, we explored the K-Nearest Neighbor (K-NN) classifier. While simple, it was fundamentally flawed for image classification due to its $O(N)$ prediction time and reliance on raw pixel distance.

We now introduce the most important building block in deep learning: the **Parametric Approach**, specifically the **Linear Classifier**. Unlike K-NN, which memorizes the entire dataset, a parametric model summarizes the knowledge of the training data into a set of parameters (weights). Once trained, the training data can be discarded!

## 4.1 The Parametric Approach Overview

In a parametric model, we define a function $f(\mathbf{x}, \mathbf{W})$ that takes the input image $\mathbf{x}$ and a set of parameters $\mathbf{W}$ (weights) and outputs the predicted class scores.

The simplest possible function is a linear mapping:

$$ f(\mathbf{x}, \mathbf{W}) = \mathbf{W}\mathbf{x} + \mathbf{b} $$

Where:
*   $\mathbf{x}$ is the input image tensor flattened into a single column vector.
*   $\mathbf{W}$ is the weight matrix (the parameters we will eventually learn).
*   $\mathbf{b}$ is the bias vector. It allows the function to shift the output scores independently of the input data $\mathbf{x}$ (e.g., if cats are much more common in our dataset, the cat bias might be automatically learned to be higher).

## 4.2 The Algebraic Viewpoint

Let's break down the exact dimensions of this matrix multiplication for the CIFAR-10 dataset (which contains $32 \times 32 \times 3$ images and 10 classes).

1.  **Flatten the Image:** The input image $\mathbf{x}$ is originally $32 \times 32 \times 3$. We flatten it into a single column vector of size $3072 \times 1$ (since $32 \times 32 \times 3 = 3072$).
2.  **Output Scores:** We want 10 scores (one for each class). Thus, our output $f(\mathbf{x}, \mathbf{W})$ must be a $10 \times 1$ vector.
3.  **The Weight Matrix:** By the rules of matrix multiplication, for $\mathbf{W}\mathbf{x}$ to produce a $10 \times 1$ vector when $\mathbf{x}$ is $3072 \times 1$, the matrix $\mathbf{W}$ must be **$10 \times 3072$**.
4.  **The Bias Vector:** The bias $\mathbf{b}$ is simply added to the output, so it must also be **$10 \times 1$**.

$$ \underbrace{f(\mathbf{x}, \mathbf{W})}_{10 \times 1} = \underbrace{\mathbf{W}}_{10 \times 3072} \cdot \underbrace{\mathbf{x}}_{3072 \times 1} + \underbrace{\mathbf{b}}_{10 \times 1} $$

> [!NOTE]
> Notice the incredible efficiency here! To predict the class of a new image, we only perform a single matrix multiplication and an addition. The prediction time is $O(1)$ relative to the size of the training dataset. This solves the fatal flaw of K-NN.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

# --- NUMPY IMPLEMENTATION ---
# Let's simulate a linear classifier pass for a single CIFAR-10 image

# 1. Simulate a flattened image (3072 pixels)
x = np.random.randn(3072, 1) 

# 2. Simulate random weights (10 classes x 3072 pixels)
W = np.random.randn(10, 3072)

# 3. Simulate random biases (10 classes)
b = np.random.randn(10, 1)

# 4. Compute the linear function: Wx + b
scores = W.dot(x) + b

print("--- NUMPY ---")
print("Shape of x:", x.shape)
print("Shape of W:", W.shape)
print("Shape of b:", b.shape)
print("Shape of output scores:", scores.shape)
print("Raw scores for the 10 classes:\n", scores.flatten())

# --- PYTORCH EQUIVALENT ---
print("\n--- PYTORCH ---")
# In PyTorch, nn.Linear automatically handles the Wx + b computation.
# Note: PyTorch linear layers expect inputs of shape (batch_size, features).
# So underneath, it computes: xW^T + b (transposed weight matrix).

linear_classifier = nn.Linear(in_features=3072, out_features=10)
x_torch = torch.randn(1, 3072) # Batch of 1 image

# Forward pass
scores_torch = linear_classifier(x_torch)

print(f"PyTorch input shape: {x_torch.shape}")
print(f"PyTorch output shape: {scores_torch.shape}")

## 4.3 The Visual Viewpoint: Class Templates

If we examine the matrix $\mathbf{W}$ closely, we notice it has 10 rows (one for each class), and each row has 3072 elements (exactly the size of a flattened image).

We can take a single row of $\mathbf{W}$, say the row corresponding to the "car" class, and **un-flatten** it back into a $32 \times 32 \times 3$ image. 

**What does this look like?**
Because the score for the "car" class is generated by taking the dot product between the "car row" of $\mathbf{W}$ and the input image $\mathbf{x}$, the "car row" effectively acts as a **template** (or a matched filter) for cars. The linear classifier is simply comparing the input image to 10 learned templates simultaneously via the dot product.

When trained, these templates begin to look like blurred, generalized versions of their respective classes. A trained car template might look like a red blob sitting on top of two black circles (wheels), because that pattern yields a high dot product with actual images of cars.

## 4.4 The Geometric Viewpoint: Hyperplanes

Geometrically, we can imagine the input images existing as points in a high-dimensional space (e.g., 3072 dimensions). 

Each row of the weight matrix $\mathbf{W}$ defines a hyperplane (a flat, multidimensional boundary) in this space. The linear classifier attempts to draw 10 different hyperplanes that carve up the space, separating the cats from the dogs, the ships from the planes, etc.

*   The orientation of the hyperplane is controlled by $\mathbf{W}$.
*   The position (offset from the origin) is controlled by the bias $\mathbf{b}$. Without $\mathbf{b}$, all hyperplanes would be forced to perfectly intersect the origin $(0,0,0...)$, severely restricting the classifier's flexibility.

### The Bias Trick
Sometimes, to simplify the math in proofs or code, we use the "bias trick". We append a constant $1$ to the input vector $\mathbf{x}$, making it $3073 \times 1$. We then absorb the bias $\mathbf{b}$ into the weight matrix $\mathbf{W}$ as an extra column, making it $10 \times 3073$.

This allows us to write the linear function purely as a single matrix multiplication:
$$ f(\mathbf{x}, \mathbf{W}) = \mathbf{W}\mathbf{x} $$

## 4.5 Hard Cases for Linear Classifiers

While powerful and fast, linear classifiers are inherently limited because they can only draw *straight* lines (hyperplanes). They will fail completely if the data is not linearly separable.

Three classic hard cases:
1.  **Parity / XOR Problem:** Class 1 exists in quadrants I and III, while Class 2 exists in quadrants II and IV. No single straight line can separate them.
2.  **Circular / Ring Data:** Class 1 is clustered in the center, and Class 2 forms a ring around it. 
3.  **Multimodal Distributions:** A class (like horses) consists of two distinct visual modes (e.g., horses facing left vs horses facing right). A linear classifier can only learn *one* template per class. If it tries to merge a left-facing horse and a right-facing horse, it creates a useless two-headed horse template.

To solve these hard cases, we will eventually need the non-linear capabilities of Deep Neural Networks.

---
### Summary of Section 4
*   **Concepts Introduced:** Parametric approach, Linear Classifier, Bias Trick, Algebraic viewpoint (dimensions), Visual viewpoint (templates), Geometric viewpoint (hyperplanes), Hard Cases (XOR, Multimodal).
*   **Equations Derived:** $f(\mathbf{x}, \mathbf{W}) = \mathbf{W}\mathbf{x} + \mathbf{b}$
*   **Notation Introduced:** $\mathbf{W}$ (Weight matrix), $\mathbf{b}$ (Bias vector), $f(\mathbf{x}, \mathbf{W})$ (Scoring function).
*   **Dependencies for Next Section:** We now have a mechanism that takes an image and outputs 10 arbitrary scores. We need a mathematical way to evaluate how "good" or "bad" these scores are, which brings us to Loss Functions.

# Section 5: Loss Functions (Softmax and Cross-Entropy)

In the previous section, we built a Linear Classifier capable of generating 10 arbitrary scores (logits) for any given input image. 

However, when we initialize our weight matrix $\mathbf{W}$ randomly, these scores will be completely meaningless. To train the model, we need a mathematical way to quantify how "unhappy" we are with the current scores. This measurement is called the **Loss Function** (or Objective Function). Our ultimate goal in machine learning is to *minimize* this loss.

## 5.1 The Need for Probabilities

Let's say our linear classifier produces the following raw scores for an image of a cat:
*   Cat: 3.2
*   Dog: 5.1
*   Ship: -1.7

These numbers are unbounded and difficult to interpret. It thinks "Dog" is the highest, but what does 5.1 mean in absolute terms? To make these scores mathematically manageable and interpretable, we want to convert them into a **probability distribution** over the classes. 

A valid probability distribution must satisfy two rules:
1.  All values must be positive ($P \geq 0$).
2.  All values must sum to 1 ($\sum P = 1$).

## 5.2 The Softmax Function

The **Softmax function** takes a vector of arbitrary real-valued scores (often called *logits*) and squashes them into a valid probability distribution.

Let $s_k$ be the raw score for class $k$. The probability assigned to class $k$ is given by:

$$ P(Y=k | X=x_i) = \frac{e^{s_k}}{\sum_{j} e^{s_j}} $$

**Step-by-step breakdown:**
1.  **Exponentiate:** We take $e^{s_k}$ for every score. Because $e^x$ is always positive, this guarantees all our numbers are now strictly positive, satisfying the first rule of probabilities. It also heavily penalizes negative scores and amplifies positive scores.
2.  **Normalize:** We divide each exponentiated score by the sum of all exponentiated scores. This guarantees that the final outputs sum to 1, satisfying the second rule.

If we apply Softmax to our earlier example:
*   $e^{3.2} \approx 24.5$
*   $e^{5.1} \approx 164.0$
*   $e^{-1.7} \approx 0.18$
*   **Sum** $\approx 188.68$

Normalized probabilities:
*   $P(\text{Cat}) = 24.5 / 188.68 \approx 0.13$
*   $P(\text{Dog}) = 164.0 / 188.68 \approx 0.87$
*   $P(\text{Ship}) = 0.18 / 188.68 \approx 0.001$

The model is 13% confident the image is a cat, and 87% confident it's a dog.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

# --- Numpy Softmax ---
scores = np.array([3.2, 5.1, -1.7])

# Step 1: Exponentiate
exp_scores = np.exp(scores)

# Step 2: Normalize
probabilities = exp_scores / np.sum(exp_scores)

print("--- NUMPY ---")
print(f"Raw scores: {scores}")
print(f"Probabilities: {probabilities}")
print(f"Sum of probabilities: {np.sum(probabilities):.2f}\n")

# --- PyTorch Softmax ---
scores_torch = torch.tensor([3.2, 5.1, -1.7])
# dim=0 because this is just a 1D tensor
probs_torch = F.softmax(scores_torch, dim=0) 

print("--- PYTORCH ---")
print(f"PyTorch Probabilities: {probs_torch}")

## 5.3 Cross-Entropy Loss

Now that we have probabilities, how do we define the loss? We want the probability of the *correct* class to be as close to 1 as possible (and consequently, all incorrect classes close to 0).

Let $y_i$ be the true correct class label. We want to **maximize** $P(Y=y_i | X=x_i)$.

In optimization, we conventionally *minimize* loss. To turn a maximization problem into a minimization problem, we negate it. Furthermore, taking the natural logarithm ($\log$) of the probability makes the math vastly easier (especially for computing gradients later) without changing the location of the minimum.

Thus, we arrive at the **Cross-Entropy Loss** $L_i$ for a single example:

$$ L_i = -\log \left( \frac{e^{s_{y_i}}}{\sum_{j} e^{s_j}} \right) $$

Or more simply:
$$ L_i = -\log(P_{y_i}) $$

Where $P_{y_i}$ is the Softmax probability assigned to the correct class.

> [!NOTE]
> If the model is 100% confident in the correct class ($P = 1$), the loss is $-\log(1) = 0$. If it is completely wrong ($P \to 0$), the loss approaches $\infty$.

## 5.4 Information Theory Perspective

Why is it called "Cross-Entropy"? In Information Theory, the **Kullback-Leibler (KL) Divergence** measures how different a predicted probability distribution $Q$ is from a true target distribution $P$.

Our true distribution $P$ is a "one-hot" vector (all probability mass is on the correct class, e.g., `[1, 0, 0]`). Our predicted distribution $Q$ is the output of the Softmax function.

Minimizing the KL Divergence between the one-hot target distribution and our predicted distribution mathematically simplifies down to the exact same formula: $-\log(P_{y_i})$. Thus, Cross-Entropy Loss is equivalent to forcing our predicted distribution to match the ground-truth one-hot distribution.

## 5.5 Loss Characteristics and Sanity Checks

Understanding the mathematical bounds of your loss function is critical for debugging neural networks.

### Min and Max Values
*   **Minimum Loss:** $0$. Occurs when the probability of the correct class is exactly $1.0$.
*   **Maximum Loss:** $\infty$. Occurs when the probability of the correct class is exactly $0.0$.

### The Initialization Sanity Check
When you first initialize a linear classifier with small random weights, all scores $s_j$ will be roughly equal, hovering near $0$. 
If all scores are equal, the Softmax probabilities will be roughly uniform. If you have $C$ classes, the probability of the correct class will be $P_{y_i} \approx \frac{1}{C}$.

Therefore, your loss at step 0 (initialization) should always be:
$$ L \approx -\log\left(\frac{1}{C}\right) = \ln(C) $$

For CIFAR-10 ($C=10$), the expected initial loss is $\ln(10) \approx 2.3$.

**Debugging Tip:** If you build a neural network for CIFAR-10 and your initial loss is $15.0$ or $0.1$, you know you have a bug in your code before you even start training!

In [ ]:
import numpy as np

C = 10 # CIFAR-10 classes
expected_loss = -np.log(1/C)
print(f"Expected initial loss for {C} classes: {expected_loss:.4f}")

# Let's verify with code!
# Small random weights ~ 0
W_init = np.random.randn(10, 3072) * 0.0001 
x_dummy = np.random.randn(3072, 1)

scores_init = W_init.dot(x_dummy).flatten()
print(f"Initial raw scores: {scores_init[:4]}...") # Since weights are tiny, scores are all ~0

# Softmax
exp_scores_init = np.exp(scores_init)
probs_init = exp_scores_init / np.sum(exp_scores_init)

# Assume the correct class is randomly index 3
correct_class = 3
p_correct = probs_init[correct_class]

initial_loss = -np.log(p_correct)
print(f"Calculated initial loss: {initial_loss:.4f}")

---
### Summary of Section 5
*   **Concepts Introduced:** Loss/Objective functions, Softmax function, Cross-Entropy Loss, Logits to Probabilities, KL Divergence equivalence, Initialization Sanity Checks.
*   **Equations Derived:** 
    *   Softmax: $P(Y=k | X=x_i) = \frac{e^{s_k}}{\sum_{j} e^{s_j}}$
    *   Cross-Entropy: $L_i = -\log(P_{y_i})$
*   **Notation Introduced:** $L_i$ (Loss for a single example), $s_k$ (Logit score for class $k$), $P_{y_i}$ (Probability of correct class).
*   **Conclusion:** We have successfully built the architecture to generate predictions and evaluate those predictions mathematically. The final remaining piece (covered in the next lecture) is **Optimization**: computing gradients to update $\mathbf{W}$ and actually minimize this loss!